# FRED Macro Briefing Crew

This notebook demonstrates a three-agent CrewAI workflow that uses four FRED macroeconomic series:

- Consumer Price Index (`CPIAUCSL`)
- Unemployment Rate (`UNRATE`)
- Federal Funds Rate (`FEDFUNDS`)
- 10-Year Minus 2-Year Treasury Spread (`T10Y2Y`)

The Researcher collects and profiles the data, the Analyst calculates deterministic statistics, and the Writer produces a concise macroeconomic briefing.

In [ ]:
from pathlib import Path
import json
import runpy

import pandas as pd
from IPython.display import Image, Markdown, display

import fred_tools as ft
import multi_snapshot_evaluation as mse
import reporting

## Load a reproducible FRED snapshot

The notebook uses the original November 4 snapshot when it is available. In a clean checkout, it prepares the July 2023 historical example by downloading the current FRED series and removing observations after the cutoff. This preparation does not call a language model.


In [ ]:
ORIGINAL_SNAPSHOT_DIR = Path("data/fred/2026-11-04")
HISTORICAL_SNAPSHOT_DIR = Path(
    "data/fred/historical/2023-07-31"
)

if ORIGINAL_SNAPSHOT_DIR.exists():
    SNAPSHOT_DIR = ORIGINAL_SNAPSHOT_DIR
elif HISTORICAL_SNAPSHOT_DIR.exists():
    SNAPSHOT_DIR = HISTORICAL_SNAPSHOT_DIR
else:
    SNAPSHOT_DIR = mse.prepare_snapshot(
        "2023-07-31",
        Path("data/fred/historical"),
    )

print(f"Using snapshot: {SNAPSHOT_DIR}")

profiles = ft.profile_fred_snapshot(SNAPSHOT_DIR)
profile_table = pd.DataFrame(profiles).T[
    [
        "series_name",
        "latest_date",
        "latest_value",
        "prior_year_date",
        "prior_year_value",
        "missing_values",
    ]
]
display(profile_table)


## Calculate deterministic macroeconomic statistics

The Analyst tool uses pandas-based calculations rather than asking the language model to perform arithmetic.

In [ ]:
statistics = ft.calculate_macro_statistics(SNAPSHOT_DIR)

statistics_table = pd.DataFrame(
    {
        "Statistic": statistics.keys(),
        "Value": statistics.values(),
    }
)
display(statistics_table)

## Visualize the four FRED series

In [ ]:
OUTPUT_DIR = Path("outputs")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

CHART_PATH = OUTPUT_DIR / "fred_macro_series.png"
reporting.create_fred_chart(
    SNAPSHOT_DIR,
    CHART_PATH,
)

display(Image(filename=str(CHART_PATH)))

## Review the generated macroeconomic briefing

In [ ]:
BRIEFING_PATH = OUTPUT_DIR / (
    f"fred_macro_briefing_{SNAPSHOT_DIR.name}.md"
)

if BRIEFING_PATH.exists():
    display(
        Markdown(
            BRIEFING_PATH.read_text(encoding="utf-8")
        )
    )
else:
    print(
        "Run crewai.example.py for the selected snapshot "
        "to generate its briefing."
    )


## Compare the CrewAI workflow with the single-LLM baseline

In [ ]:
EVALUATION_PATH = OUTPUT_DIR / "evaluation_report.json"

if EVALUATION_PATH.exists():
    evaluation = json.loads(
        EVALUATION_PATH.read_text(encoding="utf-8")
    )
    comparison = pd.DataFrame(
        {
            "Method": [
                "Three-agent CrewAI workflow",
                "Single-LLM baseline",
            ],
            "Correct required claims": [
                evaluation["crew"]["correct_claims"],
                evaluation["single_llm_baseline"][
                    "correct_claims"
                ],
            ],
            "Submitted claims": [
                evaluation["crew"]["submitted_claims"],
                evaluation["single_llm_baseline"][
                    "submitted_claims"
                ],
            ],
            "Required claims": [
                evaluation["crew"]["required_claims"],
                evaluation["single_llm_baseline"][
                    "required_claims"
                ],
            ],
            "Accuracy": [
                evaluation["crew"]["accuracy"],
                evaluation["single_llm_baseline"]["accuracy"],
            ],
            "Completeness": [
                evaluation["crew"]["completeness"],
                evaluation["single_llm_baseline"][
                    "completeness"
                ],
            ],
            "Required-claim score": [
                evaluation["crew"]["fraction_correct"],
                evaluation["single_llm_baseline"][
                    "fraction_correct"
                ],
            ],
        }
    )
    display(comparison)
else:
    print("Run evaluation.py to generate the comparison.")

## Historical Multi-Snapshot Experiment

The robustness experiment uses four observation cutoffs representing different economic conditions. The runner downloads the current FRED series and removes observations after each cutoff.

The following cell reads only cached snapshots. It does not download data or make language-model calls. These snapshots are not revision-aware ALFRED vintages.


In [ ]:
HISTORICAL_DATA_DIR = Path("data/fred/historical")
historical_rows = []

for cutoff in mse.DEFAULT_CUTOFFS:
    snapshot_dir = HISTORICAL_DATA_DIR / cutoff
    expected_files = [
        snapshot_dir / f"{series_id}.csv"
        for series_id in ft.FRED_SERIES
    ]
    if all(path.exists() for path in expected_files):
        summary = mse.summarize_snapshot(
            cutoff,
            snapshot_dir,
        )
        historical_rows.append(
            {
                "Cutoff": summary["cutoff"],
                "CPI date": summary["latest_dates"]["CPIAUCSL"],
                "Unemployment date": summary["latest_dates"][
                    "UNRATE"
                ],
                "Fed funds date": summary["latest_dates"][
                    "FEDFUNDS"
                ],
                "Yield spread date": summary["latest_dates"][
                    "T10Y2Y"
                ],
                "Inflation (%)": summary["inflation_yoy_pct"],
                "Unemployment (%)": summary[
                    "unemployment_rate_pct"
                ],
                "Fed funds (%)": summary["fed_funds_rate_pct"],
                "Yield spread (pp)": summary["yield_spread_pp"],
                "Yield curve sign": summary["yield_curve_sign"],
            }
        )

if historical_rows:
    display(pd.DataFrame(historical_rows))
else:
    print(
        "Historical snapshots are not cached. Run "
        "`python multi_snapshot_evaluation.py` to prepare "
        "them without making LLM calls."
    )


## Review the Multi-Snapshot Evaluation

When cached results are available, the following cell compares the CrewAI workflow and single-LLM baseline across all four snapshots.

Both methods produced all 48 required claims correctly in the completed experiment. This is an honest tie: the experiment did not show a numerical-accuracy improvement from CrewAI over the baseline.


In [ ]:
AGGREGATE_PATH = Path(
    "outputs/multi_snapshot/aggregate_report.json"
)

if AGGREGATE_PATH.exists():
    aggregate = json.loads(
        AGGREGATE_PATH.read_text(encoding="utf-8")
    )
    labels = {
        "crew": "Three-agent CrewAI workflow",
        "single_llm_baseline": "Single-LLM baseline",
    }
    aggregate_rows = []
    for method, label in labels.items():
        metrics = aggregate["methods"][method]
        aggregate_rows.append(
            {
                "Method": label,
                "Snapshots": metrics["snapshot_count"],
                "Correct": metrics["total_correct_claims"],
                "Required": metrics["total_required_claims"],
                "Mean accuracy": metrics["mean_accuracy"],
                "Mean completeness": metrics[
                    "mean_completeness"
                ],
                "Pooled score": metrics[
                    "pooled_fraction_correct"
                ],
            }
        )
    display(pd.DataFrame(aggregate_rows))
else:
    print(
        "The aggregate result is not cached. Run "
        "`python multi_snapshot_evaluation.py --run-llm` "
        "after configuring the API key and model."
    )


## Optionally run the three-agent crew

LLM execution is disabled by default so the notebook can be tested without making paid or rate-limited API calls. Set `RUN_LLM = True` only when a Gemini API key and `CREWAI_MODEL` are available.

In [ ]:
RUN_LLM = False

if RUN_LLM:
    example_module = runpy.run_path(
        "crewai.example.py",
        run_name="crewai_example",
    )
    result = example_module["run_fred_crew"](
        data_dir=str(SNAPSHOT_DIR.parent),
        download_date=SNAPSHOT_DIR.name,
        observation_cutoff=SNAPSHOT_DIR.name,
        llm=None,
        verbose=False,
    )
    display(Markdown(str(result)))
else:
    print(
        "LLM execution is disabled. Set RUN_LLM = True "
        "to run the three-agent crew."
    )
